In [4]:

import json
import pandas as pd
from tqdm import tqdm
import numpy as np
from collections import defaultdict
def read_from_jsonl(filename):
    data = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            data.append(item)
    return data
def load_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        return data
    except FileNotFoundError:
        print(f"文件未找到: {file_path}")
        return None
    except json.JSONDecodeError:
        print(f"文件格式错误: {file_path}")
        return None
    except Exception as e:
        print(f"加载 JSON 文件时发生错误: {e}")
        return None

In [6]:
# 读取数据
df_examples = pd.read_parquet('../data/esci-data/shopping_queries_dataset_examples.parquet')
df_products = pd.read_parquet('../data/esci-data/shopping_queries_dataset_products.parquet')
df_sources = pd.read_csv("../data/esci-data/shopping_queries_dataset_sources.csv")


df_examples_products = pd.merge(
    df_examples,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)

lang='jp'
# 筛选出 small_version 为 1 的数据
df_task_1 = df_examples_products[df_examples_products['small_version'] == 1]
df_task_1_us = df_task_1[(df_task_1['product_locale'] == lang)]
# 筛选出 train 数据集并且 product_locale 为 'us' 的数据
df_task_1_train_us = df_task_1[(df_task_1["split"] == "train") & (df_task_1['product_locale'] == lang)]
df_task_1_test_us = df_task_1[(df_task_1["split"] == "test") & (df_task_1['product_locale'] == lang)]

train_query_docs = df_task_1_train_us[['query', 'product_id', 'esci_label']].drop_duplicates()
test_query_docs = df_task_1_test_us[['query', 'product_id', 'esci_label']].drop_duplicates()
# 转换为字典格式
train_data_dict = train_query_docs.groupby('query').apply(
    lambda x: list(zip(x['product_id'], x['esci_label']))
).to_dict()
test_data_dict = test_query_docs.groupby('query').apply(
    lambda x: list(zip(x['product_id'], x['esci_label']))
).to_dict()

/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_53430/1016564064.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data_dict = train_query_docs.groupby('query').apply(
/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_53430/1016564064.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_data_dict = test_query_docs.groupby('query').apply(


In [7]:

product_info_dict = df_task_1_us.set_index('product_id').T.to_dict()

indexed_product_info_dict = {}
product_id_to_index = {}

for index, (product_id, product_info) in enumerate(product_info_dict.items()):
    indexed_product_info_dict[index] = product_info
    product_id_to_index[product_id] = index

/var/folders/80/yk28jv5n41x_7lhz5w2ykd2h0000gn/T/ipykernel_53430/1318033569.py:1: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  product_info_dict = df_task_1_us.set_index('product_id').T.to_dict()


In [8]:
len(product_id_to_index)

233850

In [10]:
from collections import defaultdict
from tqdm import tqdm

esci_label_rel = {"E":3, "S":2, "C":1, "I":0}

def create_query_doc_rels(train_data_dict, product_id_to_index):

    pairs = []
    for query in train_data_dict:
        for (product_id, esci_label) in train_data_dict[query]:
            if esci_label != 'I':
                pairs.append((query, product_id_to_index[product_id], esci_label_rel[esci_label]))

    
    with open(f"../data/esci_{lang}/test_qrels.txt", 'w') as f:
        for (q, idx, rel) in pairs:
            f.write(f"{q}\t{idx}\t{rel}\n")

create_query_doc_rels(test_data_dict, product_id_to_index)


In [ ]:
from collections import defaultdict
from tqdm import tqdm

esci_label_rel = {"E":3, "S":2, "C":1, "I":0}

def create_query_doc_rels(train_data_dict, product_id_to_index):
    pairs = []
    for query in train_data_dict:
        for (product_id, esci_label) in train_data_dict[query]:
            if esci_label != 'I':
                pairs.append((query, product_id_to_index[product_id], esci_label_rel[esci_label]))

    
    with open(f"../data/esci_{lang}/qrels.txt", 'w') as f:
        for (q, idx, rel) in pairs:
            f.write(f"{q}\t{idx}\t{rel}\n")

create_query_doc_rels(train_data_dict, product_id_to_index)


In [12]:
query2idx = {}
cnt = 0
for query in train_data_dict.keys():
    if query not in query2idx:
        query2idx[query] = cnt
        cnt += 1
print(cnt)
with open(f"../data/esci_{lang}/train_query_to_index.json", 'w') as f:
    json.dump(query2idx, f, ensure_ascii=False)

7284
